# Agent 2 — Data Ingestion

**What this notebook does:**  
1. Loads our 170-company universe from the STOXX 600 outperformers Excel file  
2. Filters the four professor CSV files to only those companies  
3. Downloads stock prices from Yahoo Finance  
4. Merges everything into one master dataset  

**How to present this to investors:**  
> *Our universe is the 170 STOXX Europe 600 companies that outperformed the index on both a 5-year and 10-year total return basis. We then filter these through four Bloomberg/professor-provided ESG datasets and apply sector exclusions, producing a clean master dataset for our 13-agent pipeline.*

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import openpyxl
import yfinance as yf
from datetime import date

TODAY = str(date.today())
DATA_DIR   = "../data/provided"
MARKET_DIR = "../data/market"
STOXX_FILE = "../STOXX600_Outperformers_5Y_10Y.xlsx"
os.makedirs(MARKET_DIR, exist_ok=True)
print(f"Run date: {TODAY}")

## Step 1 — Build the 170-company universe from the STOXX 600 outperformers file

In [ ]:
# Bloomberg exchange code → Yahoo Finance suffix
EXCHANGE_MAP = {
    'DE': '.DE', 'SE': '.ST', 'NL': '.AS', 'GB': '.L',
    'CH': '.SW', 'FR': '.PA', 'IT': '.MI', 'ES': '.MC',
    'NO': '.OL', 'FI': '.HE', 'DK': '.CO', 'BE': '.BR',
    'AT': '.VI', 'PL': '.WA', 'PT': '.LS', 'IE': '.IR',
    'LU': '.LU',
}

def stoxx_to_yf(stoxx_ticker):
    """Convert Bloomberg STOXX ticker (e.g. 'ASML-NL', 'LAGR.B-SE') to Yahoo Finance ticker."""
    if not stoxx_ticker or '-' not in str(stoxx_ticker):
        return None
    ticker_part, exch_code = str(stoxx_ticker).rsplit('-', 1)
    suffix = EXCHANGE_MAP.get(exch_code, f'.{exch_code}')
    ticker_part = ticker_part.replace('.', '-')  # LAGR.B → LAGR-B
    return f"{ticker_part}{suffix}"

wb = openpyxl.load_workbook(STOXX_FILE)

# Sheet 1: 170 outperformers (rank, symbol, company, 5Y TR, 10Y TR)
ws170 = wb['170 Outperformers ']
outperformers = []
for row in ws170.iter_rows(min_row=2, max_row=171, values_only=True):
    if row[2]:
        outperformers.append({
            'rank': row[0], 'company': row[2],
            'return_5y_pct': row[3], 'return_10y_pct': row[4]
        })
df_universe = pd.DataFrame(outperformers)

# Sheet 2: Full STOXX 600 list → name → Bloomberg ticker (for Yahoo Finance conversion)
ws600 = wb['Stoxx 600 List']
name_to_stoxx = {}
for row in ws600.iter_rows(min_row=7, max_row=611, values_only=True):
    if row[0] and row[1]:
        name_to_stoxx[row[0].lower().strip()] = row[1]

df_universe['stoxx_ticker'] = df_universe['company'].str.lower().str.strip().map(name_to_stoxx)
df_universe['yf_ticker']    = df_universe['stoxx_ticker'].apply(stoxx_to_yf)

print(f"Universe: {len(df_universe)} companies")
print(f"Yahoo Finance tickers resolved: {df_universe['yf_ticker'].notna().sum()}/170")
df_universe.head(10)

## Step 2 — Load equityBicsV2 and filter to our universe

In [ ]:
JOIN_KEY = "idBbCompany"

def normalize(s):
    s = str(s).lower().strip()
    s = re.sub(r'\b(n\.v\.|nv|s\.a\.|sa|s\.p\.a\.|spa|plc|ag|ab|asa|oyj|se|ltd\.|ltd|group|holding|holdings|class [ab])\b', '', s)
    s = re.sub(r'[^\w\s]', '', s)
    return re.sub(r'\s+', ' ', s).strip()

print("Loading equityBicsV2.csv (takes ~15 seconds)...")
df_ids = pd.read_csv(f"{DATA_DIR}/equityBicsV2.csv", low_memory=False)
print(f"Loaded: {len(df_ids):,} rows | {df_ids[JOIN_KEY].nunique():,} unique companies")

# Build lookup tables from equityBicsV2
df_ids['_norm'] = df_ids['idBbGlobalCompanyName'].fillna('').apply(normalize)
norm_to_id   = df_ids.drop_duplicates('_norm').set_index('_norm')[JOIN_KEY].to_dict()
ticker_to_id = df_ids.dropna(subset=['ticker']).drop_duplicates('ticker').set_index('ticker')[JOIN_KEY].to_dict()

# Match each of the 170 companies to an idBbCompany
matched, unmatched = {}, []
for _, row in df_universe.iterrows():
    company = row['company']
    stoxx_t = row.get('stoxx_ticker', '')
    found = False

    # Try ticker variants from STOXX 600 list
    if stoxx_t:
        root = stoxx_t.split('-')[0]
        for variant in [root, root.replace('.',''), root.split('.')[0]]:
            if variant in ticker_to_id:
                matched[company] = ticker_to_id[variant]
                found = True
                break

    # Fall back to normalized name
    if not found:
        n = normalize(company)
        if n in norm_to_id:
            matched[company] = norm_to_id[n]
            found = True

    if not found:
        unmatched.append(company)

df_universe[JOIN_KEY] = df_universe['company'].map(matched)
target_ids = set(matched.values())

print(f"\nMatched: {len(matched)}/170 companies → {len(target_ids)} unique idBbCompany IDs")
print(f"Unmatched ({len(unmatched)}): {unmatched}")

In [ ]:
# Filter equityBicsV2 to universe and keep ONE row per company (the primary/common equity)
df_filtered = df_ids[df_ids[JOIN_KEY].isin(target_ids)].copy()
print(f"Rows for our universe: {len(df_filtered):,}")

# Prefer common equity rows (securityTyp == 'Common Stock' or similar)
if 'securityTyp' in df_filtered.columns:
    common = df_filtered[df_filtered['securityTyp'].str.contains('Common|EQS', case=False, na=False)]
    df_company = common.drop_duplicates(subset=JOIN_KEY, keep='first') if len(common) > 0 else df_filtered.drop_duplicates(subset=JOIN_KEY, keep='first')
else:
    df_company = df_filtered.drop_duplicates(subset=JOIN_KEY, keep='first')

# Attach the universe metadata (rank, 5Y/10Y return, Yahoo Finance ticker)
df_company = df_company.merge(
    df_universe[[JOIN_KEY, 'rank', 'company', 'return_5y_pct', 'return_10y_pct', 'yf_ticker']].dropna(subset=[JOIN_KEY]),
    on=JOIN_KEY, how='left'
)

print(f"Companies in filtered dataset: {len(df_company)}")
print(f"Columns: {list(df_company.columns[:10])} ...")
df_company[['idBbGlobalCompanyName', 'rank', 'return_10y_pct', 'yf_ticker']].sort_values('rank').head(10)

## Step 3 — Apply sector exclusions (gambling and defense)

In [ ]:
# Exclusion keywords in BICS classification columns
EXCLUDE_KEYWORDS = [
    'casino', 'gaming', 'gambling', 'lottery',       # gambling
    'defense', 'defence', 'weapons', 'ammunition',   # defense
    'tobacco',                                        # tobacco (optional — mandate-based)
]

class_cols = [c for c in df_company.columns if 'classificationLevelName' in c]
print(f"BICS classification columns: {class_cols}")

def is_excluded(row):
    for col in class_cols:
        val = str(row.get(col, '')).lower()
        if any(kw in val for kw in EXCLUDE_KEYWORDS):
            return True
    return False

df_company['excluded'] = df_company.apply(is_excluded, axis=1)
excluded = df_company[df_company['excluded']]
df_company = df_company[~df_company['excluded']].copy()

print(f"Excluded companies ({len(excluded)}):")
for _, row in excluded.iterrows():
    reason = next((str(row[c]) for c in class_cols if any(kw in str(row[c]).lower() for kw in EXCLUDE_KEYWORDS)), 'unknown')
    print(f"  - {row.get('idBbGlobalCompanyName', 'N/A')} | Reason: {reason}")

print(f"\nUniverse after exclusions: {len(df_company)} companies")

## Step 4 — Load and merge ESG environmental + social data

In [ ]:
print("Loading ESG environmental + social data...")
df_esg = pd.read_csv(f"{DATA_DIR}/esgEnvironmentalSocialConsolidatedV4.csv", low_memory=False)
print(f"Loaded: {len(df_esg):,} rows x {len(df_esg.columns)} columns")

# Filter to our universe
df_esg = df_esg[df_esg[JOIN_KEY].isin(target_ids)]
print(f"After universe filter: {len(df_esg)} rows")

# Keep most recent reporting period per company
if 'latestPeriodEndCsr' in df_esg.columns and df_esg[JOIN_KEY].duplicated().any():
    df_esg = df_esg.sort_values('latestPeriodEndCsr', ascending=False).drop_duplicates(JOIN_KEY, keep='first')
    print(f"After keeping most recent year per company: {len(df_esg)} rows")

print(f"ESG rows matched to universe: {len(df_esg)}")

## Step 5 — Load and merge governance data

In [ ]:
print("Loading governance data...")
df_gov = pd.read_csv(f"{DATA_DIR}/esgGovernanceConsolidatedV4.csv", low_memory=False)
print(f"Loaded: {len(df_gov):,} rows x {len(df_gov.columns)} columns")

df_gov = df_gov[df_gov[JOIN_KEY].isin(target_ids)]

if 'latestPeriodEndCsr' in df_gov.columns and df_gov[JOIN_KEY].duplicated().any():
    df_gov = df_gov.sort_values('latestPeriodEndCsr', ascending=False).drop_duplicates(JOIN_KEY, keep='first')

print(f"Governance rows matched to universe: {len(df_gov)}")

## Step 6 — Load and merge EU Taxonomy data

In [ ]:
print("Loading EU Taxonomy data...")
df_tax = pd.read_csv(f"{DATA_DIR}/legalEntityEuTaxonomy.csv", low_memory=False)
print(f"Loaded: {len(df_tax):,} rows x {len(df_tax.columns)} columns")

df_tax = df_tax[df_tax[JOIN_KEY].isin(target_ids)]

if 'latestPeriodEndCsr' in df_tax.columns and df_tax[JOIN_KEY].duplicated().any():
    df_tax = df_tax.sort_values('latestPeriodEndCsr', ascending=False).drop_duplicates(JOIN_KEY, keep='first')

print(f"Taxonomy rows matched to universe: {len(df_tax)}")

## Step 7 — Merge into one master table

In [ ]:
master = df_company.copy()

# Drop duplicate identifier columns from ESG/gov/tax before merging
drop_cols = ['rc', 'idBbGlobal', 'idBbGlobalCompany', 'idBbGlobalCompanyName',
             'primSecurityCompIdBbGlobal', 'eqyFundCrncy', 'cntryOfIncorporation',
             'cntryOfDomicile', 'eqyConsolidated']

for df_src, label in [(df_esg, '_esg'), (df_gov, '_gov'), (df_tax, '_tax')]:
    cols_to_drop = [c for c in drop_cols if c in df_src.columns and c != JOIN_KEY]
    df_src_clean = df_src.drop(columns=cols_to_drop, errors='ignore')
    master = master.merge(df_src_clean, on=JOIN_KEY, how='left', suffixes=('', label))
    print(f"After merging {label}: {master.shape}")

master['data_vintage'] = TODAY
print(f"\nFinal master table: {len(master)} companies x {len(master.columns)} columns")
master[['idBbGlobalCompanyName', 'rank', 'return_10y_pct', 'yf_ticker']].sort_values('rank').head(15)

## Step 8 — Download stock prices from Yahoo Finance

In [ ]:
tickers = master['yf_ticker'].dropna().unique().tolist()
print(f"Downloading prices for {len(tickers)} tickers...")
print("Sample:", tickers[:8])

price_file = f"{MARKET_DIR}/prices_{TODAY}.csv"

if os.path.exists(price_file):
    print(f"Loading cached prices from {price_file}")
    prices = pd.read_csv(price_file, index_col=0, parse_dates=True)
else:
    print("Downloading from Yahoo Finance (takes 2–4 minutes)...")
    raw = yf.download(tickers, start="2020-01-01", auto_adjust=True, progress=True)
    prices = raw["Close"]
    prices.to_csv(price_file)
    print(f"Saved to {price_file}")

downloaded = set(prices.columns.tolist())
missing_prices = [t for t in tickers if t not in downloaded]
print(f"\nPrice data: {prices.shape[0]} trading days x {prices.shape[1]} stocks")
print(f"Missing prices ({len(missing_prices)}): {missing_prices[:10]}")

## Step 9 — Data quality check

In [ ]:
missing = master.isnull().mean().mul(100).round(1)
missing_summary = missing[missing > 0].sort_values(ascending=False)

print(f"Companies in master: {len(master)}")
print(f"Total columns: {len(master.columns)}")
print(f"Columns with any missing data: {len(missing_summary)}")
print("\nTop 20 most incomplete columns:")
print(missing_summary.head(20).to_string())

## Step 10 — Save the master dataset

In [ ]:
os.makedirs("../outputs/scores", exist_ok=True)
output_path = f"../outputs/scores/master_dataset_{TODAY}.csv"
master.to_csv(output_path, index=False)
print(f"Master dataset saved: {output_path}")
print(f"Shape: {master.shape[0]} companies x {master.shape[1]} columns")
print(f"Data vintage: {TODAY}")
print(f"\nCompanies by rank (top 20):")
print(master[['idBbGlobalCompanyName', 'rank', 'return_10y_pct', 'yf_ticker']]
      .sort_values('rank').head(20).to_string(index=False))

## ✅ Notebook complete

You now have:
- A **master dataset** with all four professor CSV files merged, filtered to the 170 STOXX outperformers
- **Sector exclusions** applied (gambling, defense, tobacco)
- **5 years of stock prices** cached locally
- A **data quality report** showing which fields have missing values

**Next:** Open `agent03_data_quality.ipynb` to run the full data audit.